# pyLSTemp - Exemplo de `brightness(...)`

Este notebook demonstra como calcular **temperatura de brilho** para as bandas termais 10 e 11 usando a função pública `brightness(...)`.

A temperatura de brilho é uma etapa de pré-processamento importante antes dos métodos de LST (`single_window` e `split_window`).

Neste exemplo são usados arquivos `.tif` disponíveis na pasta `data/` deste próprio diretório.

## 1. Instalação

Execute esta célula se estiver em um ambiente novo, como Google Colab. Se a biblioteca já estiver instalada, você pode pular esta etapa.

Observação: `rasterio` é usado apenas para abrir os arquivos GeoTIFF do exemplo. Para executar localmente, abra o notebook a partir da pasta `v1.8/exemples`, pois os caminhos usam `data/...` de forma relativa.

In [ ]:
# Em Colab/Jupyter, descomente se precisar instalar.
# !pip install --quiet --upgrade git+https://github.com/daciocambraia/pyLSTemp.git rasterio

## 2. Importações

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from pylstemp import brightness, list_algorithms

import pylstemp
print("pyLSTemp version:", pylstemp.__version__)

## 3. Conferir métodos disponíveis

`list_algorithms()` mostra as famílias e métodos registrados na biblioteca. Para este notebook, a família relevante é `thermal`, onde está registrado o método `brightness`.

In [ ]:
catalog = list_algorithms()

print("Famílias disponíveis:")
print(list(catalog.keys()))

print("\nMétodos térmicos:")
print(list(catalog["thermal"].keys()))

## 4. Localizar os arquivos de entrada

Este exemplo usa as bandas termais:

- `B10`: banda termal 10.
- `B11`: banda termal 11.

A função `brightness(...)` espera a imagem em DN ou radiância escalada conforme o fluxo adotado na biblioteca. Os valores `rad_gain` e `rad_bias` podem ser informados manualmente ou lidos do cadastro padrão do sensor.

In [ ]:
data_dir = Path("data")

band_10_path = data_dir / "LC82210712016107LGN01_B10.tif"
band_11_path = data_dir / "LC82210712016107LGN01_B11.tif"

print("Band 10:", band_10_path)
print("Band 11:", band_11_path)

if not band_10_path.exists() or not band_11_path.exists():
    raise FileNotFoundError("Confira se a pasta data/ está no mesmo diretório deste notebook.")

## 5. Ler as bandas com `rasterio`

A leitura mantém também o perfil espacial (`profile`) da banda 10, útil se você quiser salvar o resultado depois.

In [ ]:
with rasterio.open(band_10_path) as src_b10:
    band_10_data = src_b10.read(1).astype(float)
    profile = src_b10.profile.copy()

with rasterio.open(band_11_path) as src_b11:
    band_11_data = src_b11.read(1).astype(float)

print("Band 10 shape:", band_10_data.shape)
print("Band 11 shape:", band_11_data.shape)
print("Band 10 dtype:", band_10_data.dtype)
print("Band 11 dtype:", band_11_data.dtype)

## 6. Criar máscara de pixels inválidos

Neste exemplo, valores `0` são tratados como pixels inválidos. A máscara é opcional, mas ajuda a evitar que bordas/nodata entrem nas estatísticas e visualizações.

In [ ]:
mask_10 = band_10_data == 0
mask_11 = band_11_data == 0

print("Pixels inválidos B10:", int(mask_10.sum()))
print("Pixels inválidos B11:", int(mask_11.sum()))

## 7. Calcular temperatura de brilho da banda 10

Parâmetros principais:

- `thermal_band`: array da banda termal.
- `band`: use `"band_10"` ou `"band_11"`.
- `sensor`: use `"landsat_8"` ou `"landsat_9"`.
- `rad_gain`: opcional; corresponde a `RADIANCE_MULT_BAND_X`.
- `rad_bias`: opcional; corresponde a `RADIANCE_ADD_BAND_X`.
- `mask`: opcional; array booleano em que `True` indica pixel inválido.

Quando `rad_gain` e `rad_bias` não são informados, a biblioteca usa os valores padrão cadastrados para o sensor.

In [ ]:
brightness_10 = brightness(
    band_10_data,
    band="band_10",
    sensor="landsat_8",
    mask=mask_10,
)

print("Brightness 10 shape:", brightness_10.shape)
print("Brightness 10 min:", np.nanmin(brightness_10))
print("Brightness 10 max:", np.nanmax(brightness_10))
print("Brightness 10 mean:", np.nanmean(brightness_10))

## 8. Calcular temperatura de brilho da banda 11

In [ ]:
brightness_11 = brightness(
    band_11_data,
    band="band_11",
    sensor="landsat_8",
    mask=mask_11,
)

print("Brightness 11 shape:", brightness_11.shape)
print("Brightness 11 min:", np.nanmin(brightness_11))
print("Brightness 11 max:", np.nanmax(brightness_11))
print("Brightness 11 mean:", np.nanmean(brightness_11))

## 9. Exemplo com `rad_gain` e `rad_bias` manuais

Use esta forma quando quiser controlar explicitamente os valores do metadado da cena.

Para Landsat 8 neste exemplo, os padrões usados pela biblioteca são:

- `RADIANCE_MULT_BAND_10 = 0.0003342`
- `RADIANCE_ADD_BAND_10 = 0.1`

In [ ]:
brightness_10_manual = brightness(
    band_10_data,
    band="band_10",
    sensor="landsat_8",
    rad_gain=0.0003342,
    rad_bias=0.1,
    mask=mask_10,
)

difference = np.nanmax(np.abs(brightness_10 - brightness_10_manual))
print("Diferença máxima entre padrão e manual:", difference)

## 10. Visualizar resultados

Para facilitar a visualização, usamos percentis para reduzir o efeito de valores extremos.

In [ ]:
def show_raster(image, title, cmap="inferno"):
    valid = image[np.isfinite(image)]
    vmin, vmax = np.nanpercentile(valid, [2, 98])
    plt.figure(figsize=(8, 6))
    plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(label="Brightness temperature (K)")
    plt.title(title)
    plt.axis("off")
    plt.show()

show_raster(brightness_10, "Brightness temperature - Band 10")
show_raster(brightness_11, "Brightness temperature - Band 11")

## 11. Visualizar uma área menor

Se a cena for grande, visualizar um recorte pode facilitar a inspeção. Ajuste os limites conforme sua imagem.

In [ ]:
height, width = brightness_10.shape

x_min = width // 3
x_max = min(width, x_min + 1000)
y_min = height // 3
y_max = min(height, y_min + 1000)

subset_10 = brightness_10[y_min:y_max, x_min:x_max]
subset_11 = brightness_11[y_min:y_max, x_min:x_max]

print("Recorte:", subset_10.shape)

show_raster(subset_10, "Brightness temperature - Band 10 subset")
show_raster(subset_11, "Brightness temperature - Band 11 subset")

## 12. Salvar resultado em GeoTIFF opcionalmente

A célula abaixo mostra como salvar a temperatura de brilho mantendo o perfil espacial do arquivo original. Ela fica comentada para evitar escrita acidental.

In [ ]:
# output_path = Path("brightness_band_10.tif")
# output_profile = profile.copy()
# output_profile.update(dtype="float32", nodata=np.nan, count=1)
#
# with rasterio.open(output_path, "w", **output_profile) as dst:
#     dst.write(brightness_10.astype("float32"), 1)
#
# print("Arquivo salvo em:", output_path)

## 13. Próximos passos

Depois de calcular `brightness_10` e `brightness_11`, você pode usar:

- `single_window(...)` com `brightness_band_10=brightness_10`.
- `split_window(...)` com `brightness_band_10=brightness_10` e `brightness_band_11=brightness_11`.
- `water_vapor(...)` se o método split-window exigir ou se você quiser estimar vapor d'água pixel a pixel.